In [ ]:
# GPT-2 FineWeb pretraining using the reviewed repository implementation
!nvidia-smi
import torch

num_gpus = max(1, torch.cuda.device_count())
print(f"GPU count: {num_gpus}")
for index in range(num_gpus):
    print(torch.cuda.get_device_name(index))


In [ ]:
# Install exactly the branch that contains this recipe.
!pip install -q uv
!git clone --depth 1 --branch spirlness/feat/gpt2-fineweb-training https://github.com/spirlness/Automodel.git Automodel
%cd Automodel
!uv sync --locked --group dev --extra fa --inexact


In [ ]:
# Produce the binary dataset with the repository-owned preprocessor.
!uv run python projects/gpt2_fineweb_500m/tools/nanogpt_data_processor.py \
  --dataset HuggingFaceFW/fineweb \
  --set-name sample-10BT \
  --output-dir /kaggle/working/fineweb \
  --max-tokens 500M


In [ ]:
# Run the same recipe as the repository. T4 uses fp16 runtime overrides;
# all architecture, optimizer grouping, loss, and data-processing code stays identical.
import torch

num_gpus = max(1, torch.cuda.device_count())
global_batch_size = 32 if num_gpus >= 2 else 16
!uv run automodel projects/gpt2_fineweb_500m/config/gpt2_fineweb_500m.yaml \
  --nproc-per-node {num_gpus} \
  --dataset.file_pattern=/kaggle/working/fineweb_max_tokens_500M/dataset.bin \
  --step_scheduler.global_batch_size={global_batch_size} \
  --step_scheduler.local_batch_size=4 \
  --model.torch_dtype=float16 \
  --distributed.mp_policy.param_dtype=torch.float16 \
  --distributed.mp_policy.output_dtype=torch.float16
